# 01 — Exploratory Data Analysis

Match data quality, class balance, team mapping coverage, and feature distributions.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_international_results, load_all_matches
from src.feature_engineering import FEATURE_COLUMNS

sns.set_theme(style="whitegrid")

In [ ]:
intl = load_international_results()
all_matches = load_all_matches()
print(f"International matches (martj42, since 2000): {len(intl)}")
print(f"With StatsBomb enrichment: {all_matches['home_xg'].notna().sum() if 'home_xg' in all_matches.columns else 0}")
print(f"Competitions:\n{intl['competition'].value_counts().head(10)}")
print(f"Date range: {intl['match_date'].min()} to {intl['match_date'].max()}")
intl.head()

In [ ]:
df = pd.read_parquet(PROJECT_ROOT / "data/processed/match_features.parquet")
print(f"Feature rows: {len(df)}")
print(f"Teams: {df['team'].nunique()}")
print(f"\nOutcome distribution:\n{df['outcome'].value_counts().sort_index()}")

df["outcome"].value_counts().sort_index().plot(kind="bar", title="Class Balance", color="steelblue")
plt.xlabel("Outcome"); plt.ylabel("Count"); plt.tight_layout(); plt.show()

In [ ]:
pre2018 = df[df["match_date"] < "2018-06-01"]
print(f"Pre-2018 training rows: {len(pre2018)}")
print(f"Elo coverage: {df['elo_diff'].notna().mean():.1%}")

null_rates = df[FEATURE_COLUMNS].isnull().mean().sort_values(ascending=False)
print("Null rates (top 10):\n", null_rates.head(10))

cols = ["elo_diff", "form_5", "form_diff_5", "xg_avg_5", "team_rest_days", "h2h_wins", "is_home"]
available = [c for c in cols if c in df.columns]
sns.heatmap(df[available + ["outcome"]].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Feature Correlations"); plt.tight_layout(); plt.show()

df["elo_diff"].hist(bins=50, color="steelblue", edgecolor="white")
plt.title("Elo diff distribution"); plt.xlabel("elo_diff"); plt.tight_layout(); plt.show()